# Session 15: Principal component analysis

An archaeologist decides which source an obsidian artefact came from: the Slovak sources (CI, in the Zemplín hills), the Hungarian sources (CII, in the Tokaj mountains), or neither. The evidence is an energy-dispersive X-ray fluorescence (ED-XRF) analysis of the artefact, ten elements, set against analyses of samples collected at the sources. This notebook reduces the ten elements to a few principal components, reads what each component measures, and tests whether an artefact belongs to a source by its distance from that source's analyses. Section 1 works the method on Dataset A, the source references. Section 2 applies it to your artefact, with the projection written by you. Section 3 compares two ways of scaling the elements before the analysis. Section 4 is the judgment cell.

The data are real: J. Petřík and L. Prokeš, "Chemical compositions of grinded obsidian artefacts from Early Eneolithic site in Nitra-Selenec (Slovakia) and reference samples of carpathian obsidians", Zenodo (2025), [doi:10.5281/zenodo.14771241](https://doi.org/10.5281/zenodo.14771241), CC BY 4.0, accessed 2026-11-05. The file `data/obsidian_data.xlsx` is the deposit as published, and `data/README.md` describes it and who has which artefact.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 15 assignment on Canvas by 1:30 pm on Tuesday, November 24. Submissions are individual. The data file is in the repository, so the graders' run reads the same data yours did. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart and holds your prompt record from section 3, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports `sys` and `Path` from the standard library, NumPy, pandas and Matplotlib, and two things new this session from scikit-learn: `StandardScaler`, which centers each column and, when asked, divides it by its standard deviation, and `PCA`, the principal component analysis. The `f` and `chi2` distributions from `scipy.stats` set the limits of the two distances in section 1, in a function the notebook gives you. The cell prints nothing when it works.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2, f
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

## Parameters

The data file and its sheet; the ten elements kept and the three left out; the sheet's source codes and what they mean, which of them are Dataset A, and which two get a model; where the sheet's two measurement campaigns divide, the reference measured in both, and how many analyses make a triplicate; the three filters; the confidence level of the limits; how many components are tried and how many every model keeps; then your artefact, set in section 2. Leave `ARTEFACT` as `CHANGE-ME` until section 2.

In [ ]:
DATA_FILE = Path("data") / "obsidian_data.xlsx"    # the Zenodo deposit, byte for byte; data/README.md
SHEET = "data"                                     # the sheet with the 246 ED-XRF analyses; the authors' own analysis reads data2
ELEMENTS = ["Rb", "Sr", "Th", "Ba", "V", "Al2O3", "SiO2", "K2O", "CaO", "Fe2O3"]   # ppm for the first five, wt% for the oxides; the ten that agree across the two campaigns
DROPPED = ["U", "MnO", "TiO2"]                     # read on different bases in the sheet's two campaigns, so not in ELEMENTS; data/README.md
SOURCES = {"CIB": "CI", "CIV": "CI", "CIIE": "CII", "CIIEM": "CII", "CIIT": "CII", "CIII": "CIII",
           "P": "artefact", "N": "artefact"}      # the sheet's Source codes: CI Slovak, CII Hungarian, CIII Ukrainian; P and N artefacts
REFERENCES = ["CI", "CII", "CIII"]                 # the source groups whose analyses are Dataset A
MODELLED = ["CI", "CII"]                           # the sources with a one-class model; CIII has three analyses, too few
CAMPAIGN_A_LAST_ROW = 102                          # rows 0 to 102 of the sheet are measurement campaign A, 103 to 245 campaign B
BOTH_CAMPAIGNS = "Vinicky2"                        # a Slovak reference measured in both campaigns, section 1's check
REPLICATES = 3                                     # analyses of one specimen in a reference triplicate, section 1's other check
SIO2_RANGE_WT_PCT = (50, 100)                      # wt%; an analysis outside it is not a normalised obsidian analysis
RED_PATTERN = "red"                                # the red obsidians, left out as the authors did
ALPHA = 0.95                                       # the confidence level of the T2 and Q limits
K_MAX = 3                                          # components tried in the leave-one-out check and at your artefact
K = 2                                              # components in every model, from section 1's leave-one-out check
FAR_OUTSIDE = 1                                    # widths of the references' score range beyond it at which section 2 says an artefact was not scaled with their scaler
ARTEFACT = "CHANGE-ME"   # your artefact's name, from data/assignments.csv by your student number, section 2

## 1. Explore: what the components measure, and how far a source reaches

Dataset A is the reference analyses: obsidian collected at the Slovak (CI), Hungarian (CII) and Ukrainian (CIII) sources and measured by the same ED-XRF method as the artefacts. The questions for this section are what each principal component of the ten elements measures, and how far from a source's analyses an analysis can sit and still belong. The first cell reads the sheet, keeps the analyses that pass the three filters, and prints what is left by source and by campaign. It ends with `BOTH_CAMPAIGNS`, a reference measured in both campaigns, which shows why three of the sheet's elements are not in `ELEMENTS`.

In [ ]:
sheet = pd.read_excel(DATA_FILE, sheet_name=SHEET)
sheet["campaign"] = np.where(sheet.index <= CAMPAIGN_A_LAST_ROW, "A", "B")     # the sheet's two measurement campaigns, by row
sheet["group"] = sheet["Source"].map(SOURCES)
no_zero = (sheet[ELEMENTS] > 0).all(axis=1)              # a zero in this deposit is a non-detect, not a measured zero
in_range = sheet["SiO2"].between(*SIO2_RANGE_WT_PCT)
not_red = ~sheet["Sample"].str.contains(RED_PATTERN, case=False)
known = sheet["group"].notna()
data = sheet[no_zero & in_range & not_red & known]
ref = data[data["group"].isin(REFERENCES)]
art = data[data["group"] == "artefact"]
print(f"sheet {SHEET!r}: {len(sheet)} analyses; failing each test (an analysis can fail more than one): a zero among the ten elements {(~no_zero).sum()}, "
      f"SiO2 outside {SIO2_RANGE_WT_PCT[0]} to {SIO2_RANGE_WT_PCT[1]} wt% {(~in_range).sum()}, red obsidian {(~not_red).sum()}, a Source code not in SOURCES {(~known).sum()}; kept {len(data)}")
print("reference analyses by source and campaign:")
print(pd.crosstab(ref["group"], ref["campaign"]).to_string())
print(f"artefact analyses {len(art)}, distinct names {art['Sample'].nunique()}, names measured {REPLICATES} times {(art.groupby('Sample').size() == REPLICATES).sum()}; "
      f"by campaign A {(art['campaign'] == 'A').sum()}, B {(art['campaign'] == 'B').sum()}")
twice = sheet[sheet["Sample"] == BOTH_CAMPAIGNS].groupby("campaign")[DROPPED + ELEMENTS].mean()
print(f"\n{BOTH_CAMPAIGNS}, one Slovak reference, mean of each campaign's analyses (the three left out first, then the ten kept; Rb to V and U in ppm, the oxides in wt%):")
print(twice.to_string(float_format=lambda v: f"{v:.3g}"))
print("campaign B over campaign A: " + ", ".join(f"{e} {twice.loc['B', e] / twice.loc['A', e]:.2f}" for e in DROPPED + ELEMENTS))

Ten elements make a ten-dimensional cloud of points, one per analysis. PCA finds the directions along which the cloud is widest: the eigenvectors of the covariance matrix, ordered by the variance each carries. Before that it centers each column on its mean. The next cell prints each element's variance over the references, then fits a PCA to the centered data and prints the share of the variance on the first three components and the loadings of the first two, the weight of each element in each component. Barium is in parts per million and the oxides in weight per cent, so the variances differ by a factor of 10⁵ or so.

In [ ]:
X_ref = ref[ELEMENTS]
print("variance of each element over the references (ppm^2 for Rb to V, wt%^2 for the oxides):")
print(X_ref.var().to_string(float_format=lambda v: f"{v:.3g}"))
scaler_center = StandardScaler(with_std=False).fit(X_ref)              # centering only
pca_center = PCA().fit(scaler_center.transform(X_ref))
print(f"\ncentered only: share of the variance on PC1 to PC3 {np.round(pca_center.explained_variance_ratio_[:3], 3).tolist()}")
loadings_center = pd.DataFrame(pca_center.components_[:2].T, index=ELEMENTS, columns=["PC1", "PC2"])
print(loadings_center.T.to_string(float_format=lambda v: f"{v:+.2f}"))

Centered only, the first component is barium, because barium's numbers are the largest, not because barium tells the sources apart. Autoscaling divides each centered column by its standard deviation, so every element enters with variance 1 and the components follow the correlations between elements instead of their units. The next cell fits the scaler and a PCA to the autoscaled references, prints the share of the variance on each component with its running total, and plots the scree. A second PCA with `K` components is the model the rest of the notebook projects onto; section 2 uses its `scaler_auto` and `pca_auto`.

In [ ]:
scaler_auto = StandardScaler().fit(X_ref)                              # centering, then division by each element's standard deviation
Z_ref = scaler_auto.transform(X_ref)
pca_full = PCA().fit(Z_ref)
share = pca_full.explained_variance_ratio_
cumulative = np.cumsum(share)
print("autoscaled: share of the variance on each component, and the running total")
for i, (s, c) in enumerate(zip(share, cumulative), start=1):
    print(f"PC{i}: {s:.3f}   running total {c:.3f}")
loadings_auto = pd.DataFrame(pca_full.components_[:K_MAX].T, index=ELEMENTS, columns=[f"PC{i}" for i in range(1, K_MAX + 1)])
print(loadings_auto.T.to_string(float_format=lambda v: f"{v:+.2f}"))
pca_auto = PCA(K).fit(Z_ref)                                            # the model: the first K components
scores_ref = pca_auto.transform(Z_ref)

fig, ax = plt.subplots()
ax.bar(range(1, len(share) + 1), share, color="lightgray", edgecolor="black", label="each component")
ax.plot(range(1, len(share) + 1), cumulative, "o-", color="black", label="running total")
ax.axvline(K + 0.5, color="gray", linestyle="--", label=f"K = {K}")
ax.set_xlabel("component (count)")
ax.set_ylabel("share of the variance (fraction)")
ax.set_title(f"Scree: {len(ref)} reference analyses, {len(ELEMENTS)} elements, autoscaled")
ax.legend()
fig.tight_layout()
plt.show()

The bars fall from 0.577 at the first component to 0.168 at the second, a running total of 0.745 at K = 2, after which each bar is under 0.1 and the total reaches 0.915 at four components. The loadings give PC1 a weight of +0.14 to +0.36 on every element, whereas PC2 sets Sr, CaO, Fe₂O₃ and V (+0.25 to +0.42) against Rb, Th, Ba, Al₂O₃, K₂O and SiO₂ (−0.20 to −0.37). The next cell plots each reference's first two scores by source, the loadings as arrows, and the correlation of each analysis's PC1 score with its mean log ratio to the average reference, one number for how much more or less of everything it reads.

In [ ]:
groups = {"CI": ("o", "black", "CI, Slovak"), "CII": ("s", "gray", "CII, Hungarian"), "CIII": ("^", "black", "CIII, Ukrainian")}
fig, (ax_scores, ax_load) = plt.subplots(1, 2, figsize=(11, 4.8))
for g, (marker, color, label) in groups.items():
    s = scores_ref[(ref["group"] == g).values]
    ax_scores.plot(s[:, 0], s[:, 1], marker, color=color, markerfacecolor="none" if g != "CIII" else color, label=f"{label} ({len(s)})")
ax_scores.axhline(0, color="lightgray", linewidth=0.8)
ax_scores.axvline(0, color="lightgray", linewidth=0.8)
ax_scores.set_xlabel(f"PC1 score (autoscaled units, {share[0]:.0%} of the variance)")
ax_scores.set_ylabel(f"PC2 score (autoscaled units, {share[1]:.0%} of the variance)")
ax_scores.set_title("Scores: the reference analyses by source")
ax_scores.legend(fontsize="small")
for element, (l1, l2) in loadings_auto[["PC1", "PC2"]].iterrows():
    ax_load.annotate("", xy=(l1, l2), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="gray"))
    ax_load.text(l1 * 1.08, l2 * 1.08, element, ha="center", va="center", fontsize="small")
ax_load.axhline(0, color="lightgray", linewidth=0.8)
ax_load.axvline(0, color="lightgray", linewidth=0.8)
ax_load.set_xlim(-0.6, 0.6)
ax_load.set_ylim(-0.6, 0.6)
ax_load.set_xlabel("PC1 loading (weight, unitless)")
ax_load.set_ylabel("PC2 loading (weight, unitless)")
ax_load.set_title("Loadings: each element's weight in PC1 and PC2")
fig.tight_layout()
plt.show()

mean_log_ratio = np.log(X_ref / X_ref.mean()).mean(axis=1).values     # one number per analysis: how much more or less of everything it reads than the average reference
print(f"correlation of each analysis's PC1 score with its mean log ratio to the reference means {np.corrcoef(scores_ref[:, 0], mean_log_ratio)[0, 1]:+.2f}, of its PC2 score {np.corrcoef(scores_ref[:, 1], mean_log_ratio)[0, 1]:+.2f}")
centre = {g: scores_ref[(ref["group"] == g).values].mean(axis=0) for g in groups}
for g in groups:
    s = scores_ref[(ref["group"] == g).values]
    print(f"{g}: PC1 from {s[:, 0].min():+.2f} to {s[:, 0].max():+.2f}, mean {centre[g][0]:+.2f}; PC2 from {s[:, 1].min():+.2f} to {s[:, 1].max():+.2f}, mean {centre[g][1]:+.2f}")

The Slovak circles run from PC1 −6.1 to +1.9 with a mean of −1.00, the Hungarian squares from −2.8 to +3.3 with a mean of +1.18, and the three Ukrainian triangles sit alone at PC2 +3.8 to +4.7, where no Slovak or Hungarian analysis passes +1.4. Every loading arrow points right: PC1 tracks each analysis's mean log ratio at +0.97, so it reads how much of everything a flake returned, and PC2, at −0.07, does not. The next cell checks the plane against two things the sheet carries: the six references measured three times, whose scatter is what one analysis can resolve, and `BOTH_CAMPAIGNS`, the same flake measured in both campaigns.

In [ ]:
by_name = ref.groupby("Sample")
triplicates = [name for name, rows in by_name if len(rows) == REPLICATES]
spread = [np.linalg.norm(scores_ref[(ref["Sample"] == name).values] - scores_ref[(ref["Sample"] == name).values].mean(axis=0), axis=1).mean()
          for name in triplicates]
print(f"{len(triplicates)} references measured {REPLICATES} times: an analysis sits {np.mean(spread):.2f} score units from its triplicate's mean, on average; "
      f"the CI and CII mean scores are {np.linalg.norm(centre['CI'] - centre['CII']):.2f} apart")
for campaign in ("A", "B"):
    s = scores_ref[((ref["Sample"] == BOTH_CAMPAIGNS) & (ref["campaign"] == campaign)).values]
    print(f"{BOTH_CAMPAIGNS} in campaign {campaign}, mean of {len(s)}: PC1 {s[:, 0].mean():+.2f}, PC2 {s[:, 1].mean():+.2f}")

A scores plot says where an artefact falls; it does not say whether it belongs. A **one-class model** of a source is a scaler and a PCA fitted to that source's analyses alone, with two distances for any new analysis: Hotelling's T², how far its scores sit from the source's centre within the model's plane, and Q, how far the analysis sits off that plane. Each has a limit that the source's own analyses exceed with probability `1 - ALPHA`. The next cell defines `one_class` and `t2_q`, then tests each CI and CII reference against its own source's model fitted without it, at 1 to `K_MAX` components, under both scalings, and fits the two models at `K`.

In [ ]:
def one_class(X_source, k, autoscale):
    """A one-class PCA model of one source: its scaler, its k-component PCA, and the ALPHA limits of T2 and Q."""
    scaler = StandardScaler(with_std=autoscale).fit(X_source)
    Z = scaler.transform(X_source)
    n = len(Z)
    full = PCA().fit(Z)
    pca = PCA(k).fit(Z)
    t2_limit = k * (n**2 - 1) / (n * (n - k)) * f.ppf(ALPHA, k, n - k)      # Jackson's limit for a new observation
    rest = full.explained_variance_[k:]                                       # the variance the k components leave out
    theta1, theta2 = rest.sum(), (rest**2).sum()
    q_limit = theta2 / theta1 * chi2.ppf(ALPHA, theta1**2 / theta2)         # Box's approximation to the Q distribution
    return dict(scaler=scaler, pca=pca, lam=pca.explained_variance_, t2_limit=t2_limit, q_limit=q_limit)


def t2_q(model, X):
    """T2 (distance within the model's plane) and Q (squared distance off it) of each row of X."""
    Z = model["scaler"].transform(X)
    T = model["pca"].transform(Z)
    t2 = (T**2 / model["lam"]).sum(axis=1)
    q = ((Z - model["pca"].inverse_transform(T))**2).sum(axis=1)
    return t2, q


def call(inside):
    """The call from a dict of source -> inside (True or False): CI, CII, both or neither."""
    yes = [s for s in MODELLED if inside[s]]
    return "both" if len(yes) == len(MODELLED) else (yes[0] if yes else "neither")


rejected = {}
for scaling, autoscale in (("centered", False), ("autoscaled", True)):
    for k in range(1, K_MAX + 1):
        out, n = 0, 0
        for s in MODELLED:
            X_s = ref.loc[ref["group"] == s, ELEMENTS]
            for i in range(len(X_s)):
                m = one_class(X_s.drop(index=X_s.index[i]), k, autoscale)
                t2, q = t2_q(m, X_s.iloc[[i]])
                out += int(t2[0] > m["t2_limit"] or q[0] > m["q_limit"])
                n += 1
        rejected[(scaling, k)] = out / n
print(f"share of the {n} CI and CII references rejected by their own source's model fitted without them (the limits expect {1 - ALPHA:.0%}):")
print(pd.Series(rejected).unstack().rename(columns=lambda k: f"k = {k}").to_string(float_format=lambda v: f"{v:.3f}"))
models = {s: one_class(ref.loc[ref["group"] == s, ELEMENTS], K, True) for s in MODELLED}
for s, m in models.items():
    print(f"{s} model, autoscaled, K = {K}: {int((ref['group'] == s).sum())} analyses, T2 limit {m['t2_limit']:.2f}, Q limit {m['q_limit']:.2f}")

**Explain.** In two or three sentences: from the loadings and the `BOTH_CAMPAIGNS` lines, which component measures composition and which measures how much of everything the flake returned, and which of the two did the measurement move? Say where the CI and CII mean scores sit on each. Then say what the rejection table says the limits do to a reference the model has not seen.

*Your answer:*

## 2. Apply: your artefact

Dataset B is one artefact, named in `data/assignments.csv`, with one or three ED-XRF analyses in the same sheet and no source label. Section 1's scaler, PCA and two models carry over unchanged. What changes is that your artefact is not one of the analyses they were fitted to, so the question is whether it belongs to a source at all.

**[TASK 1]** Find your student number in `data/assignments.csv` and set `ARTEFACT` in the Parameters cell to the name beside it, for example `"Moravany4"`. Run the Parameters cell, then the next cell. It keeps your artefact's analyses, prints them with the period the deposit gives, and sets `art_mean`, one row holding the mean of its analyses, which is what the rest of the notebook tests. Until `ARTEFACT` is set, it stops with `ValueError: no analyses named 'CHANGE-ME'`.

In [ ]:
mine = art[art["Sample"] == ARTEFACT]
if mine.empty:
    raise ValueError(f"no analyses named {ARTEFACT!r}; set ARTEFACT in Parameters from data/assignments.csv")
print(f"{ARTEFACT}: {len(mine)} analyses, period {mine['Datation'].iloc[0]}, campaign {', '.join(sorted(mine['campaign'].unique()))}")
print(mine[ELEMENTS].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
art_mean = mine[ELEMENTS].mean().to_frame().T      # one row, the columns of ELEMENTS: the artefact's composition
print("mean of the analyses:")
print(art_mean.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**[TASK 2]** In a code cell you add under this one (**+ Code**), project `art_mean` onto section 1's autoscaled model: scale it with `scaler_auto` and transform it with `pca_auto`, the two objects fitted to the references in the cell that drew the scree, and set `art_scores`, an array of shape (1, `K`). The cell after it draws your artefact on the references' scores plot and prints its distance from each source's mean score. It stops with `NameError: name 'art_scores' is not defined` until your cell has run. A scaler fitted to your artefact's own row puts the artefact at the origin, (0.00, 0.00), and the cell says so; no scaler, or centering alone, puts it far outside the references' scores, and the cell says that too. The limits cell and section 3 use `art_mean` and run without your cell, so write your plot sentence before you read them.

In [ ]:
art_scores = np.asarray(art_scores)
assert art_scores.shape == (1, K), f"art_scores has shape {art_scores.shape}, not (1, {K})"
if np.allclose(art_scores, 0, atol=1e-9):
    print("art_scores is (0.00, 0.00): the scaler was fitted to the artefact, not to the references; use scaler_auto and pca_auto from section 1")
low, high = scores_ref.min(axis=0), scores_ref.max(axis=0)
if ((art_scores[0] < low - FAR_OUTSIDE * (high - low)) | (art_scores[0] > high + FAR_OUTSIDE * (high - low))).any():
    print(f"art_scores lies more than {FAR_OUTSIDE} width of the references' scores outside them: the artefact was not scaled with scaler_auto before pca_auto")
print(f"{ARTEFACT}: PC1 {art_scores[0, 0]:+.2f}, PC2 {art_scores[0, 1]:+.2f}; " + "; ".join(f"{np.linalg.norm(art_scores[0] - centre[s]):.2f} from the {s} mean score" for s in MODELLED))

fig, ax = plt.subplots()
for g, (marker, color, label) in groups.items():
    s = scores_ref[(ref["group"] == g).values]
    ax.plot(s[:, 0], s[:, 1], marker, color=color, markerfacecolor="none" if g != "CIII" else color, label=f"{label} ({len(s)})")
ax.plot(art_scores[0, 0], art_scores[0, 1], "*", color="black", markersize=14, label=f"{ARTEFACT}, mean of {len(mine)}")
ax.set_xlabel(f"PC1 score (autoscaled units, {share[0]:.0%} of the variance)")
ax.set_ylabel(f"PC2 score (autoscaled units, {share[1]:.0%} of the variance)")
ax.set_title(f"{ARTEFACT} on the references' autoscaled scores")
ax.legend(fontsize="small")
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here, ending with the source you would call from the plot alone: CI, CII or neither.*

The plot gives a call by eye. The two models give one by the limits: the next cell computes your artefact's T² and Q against the CI model and the CII model at `K` components, autoscaled, prints each as a ratio to its limit (under 1 is inside), and makes the call, CI, CII, both or neither. It then repeats the call with 1 to `K_MAX` components, so you can see whether the number of components moves it.

In [ ]:
rows = []
calls_auto = {}
for k in range(1, K_MAX + 1):
    inside = {}
    for s in MODELLED:
        m = one_class(ref.loc[ref["group"] == s, ELEMENTS], k, True)
        t2, q = t2_q(m, art_mean)
        inside[s] = bool(t2[0] <= m["t2_limit"] and q[0] <= m["q_limit"])
        rows.append({"k": k, "source": s, "T2 / limit": t2[0] / m["t2_limit"], "Q / limit": q[0] / m["q_limit"], "inside": inside[s]})
    calls_auto[k] = call(inside)
ratios_auto = pd.DataFrame(rows)
print(f"{ARTEFACT} against each source's autoscaled model, K = {K}:")
print(ratios_auto[ratios_auto["k"] == K].drop(columns="k").to_string(index=False, float_format=lambda v: f"{v:.2f}"))
call_auto = calls_auto[K]
print(f"call at K = {K}: {call_auto}; " + ", ".join(f"k = {k}: {c}" for k, c in calls_auto.items() if k != K))

**Explain.** Before you compare: in two sentences, give your call from the plot and your call from the limits, with the ratio that decided it, and say whether 1 or 3 components would have changed it. Then compare with a neighbor: whose artefact sits nearer its limits, and did either of you get "both" or "neither"? Add one sentence on where your two answers differ and what in your data explains it. If no neighbor is ready, the notes give the solution record's line.

*Your answer:*

## 3. Analyze: centered or autoscaled

Section 1 showed that centering alone makes the first component barium, and autoscaling makes it every element at once. Both are defensible treatments of the same ten columns: centering keeps each element's weight in proportion to its spread, which an analyst who trusts the large numbers more might want, and autoscaling gives every element the same weight, which an analyst who trusts them equally wants. The question for this section is whether your artefact's call depends on the choice. The next cell fits the two models under each scaling at `K` components, prints your artefact's four ratios and its call under each, and draws it on the references' scores under each.

In [ ]:
rows = []
calls = {}
pooled = {}
for scaling, autoscale in (("centered", False), ("autoscaled", True)):
    inside = {}
    for s in MODELLED:
        m = one_class(ref.loc[ref["group"] == s, ELEMENTS], K, autoscale)
        t2, q = t2_q(m, art_mean)
        inside[s] = bool(t2[0] <= m["t2_limit"] and q[0] <= m["q_limit"])
        rows.append({"scaling": scaling, "source": s, "T2 / limit": t2[0] / m["t2_limit"], "Q / limit": q[0] / m["q_limit"], "inside": inside[s]})
    calls[scaling] = call(inside)
    scaler = StandardScaler(with_std=autoscale).fit(X_ref)
    pca = PCA(K).fit(scaler.transform(X_ref))
    pooled[scaling] = (pca.transform(scaler.transform(X_ref)), pca.transform(scaler.transform(art_mean)), pca.explained_variance_ratio_)
ratios_both = pd.DataFrame(rows)
print(f"{ARTEFACT} against each source's model, K = {K}, under each scaling:")
print(ratios_both.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
call_center = calls["centered"]
print(f"call centered: {call_center}; autoscaled: {calls['autoscaled']}; the scaling {'changes' if call_center != calls['autoscaled'] else 'does not change'} the call")
for scaling, (s_ref, s_art, evr) in pooled.items():
    print(f"{scaling} scores: {ARTEFACT} at PC1 {s_art[0, 0]:+.3g}, PC2 {s_art[0, 1]:+.3g}; the references' PC1 runs from {s_ref[:, 0].min():+.3g} to {s_ref[:, 0].max():+.3g}")
print(f"barium, the centered first component: {ARTEFACT} {art_mean['Ba'].iloc[0]:.0f} ppm; " + "; ".join(
    f"{s} references {ref.loc[ref['group'] == s, 'Ba'].min():.0f} to {ref.loc[ref['group'] == s, 'Ba'].max():.0f} ppm" for s in MODELLED))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, (scaling, (s_ref, s_art, evr)) in zip(axes, pooled.items()):
    for g, (marker, color, label) in groups.items():
        s = s_ref[(ref["group"] == g).values]
        ax.plot(s[:, 0], s[:, 1], marker, color=color, markerfacecolor="none" if g != "CIII" else color, label=f"{label} ({len(s)})")
    ax.plot(s_art[0, 0], s_art[0, 1], "*", color="black", markersize=14, label=ARTEFACT)
    unit = "ppm and wt%" if scaling == "centered" else "autoscaled units"
    ax.set_xlabel(f"PC1 score ({unit}, {evr[0]:.0%} of the variance)")
    ax.set_ylabel(f"PC2 score ({unit}, {evr[1]:.0%} of the variance)")
    ax.set_title(f"{scaling}: call {calls[scaling]}")
    ax.legend(fontsize="small")
fig.tight_layout()
plt.show()

*Write your one sentence description of the two plots here.*

**Explain.** In two or three sentences, say which treatment answers the archaeologist's question for your artefact and why, from what section 1 showed each first component measures, and whether your call depends on it: if the two calls differ, say which ratio moved across its limit and which element section 1 says that model reads. This is the evidence part (a) of the judgment cell needs.

*Your answer:*

**A fresh-context review**, today's AI practice, new this session. Save the notebook (**Ctrl+S**), so the file on disk holds your outputs. Open a new conversation that has not seen this one: in the Command Palette (**Ctrl+Shift+P**), run **Claude Code: Open in New Tab**. Type this prompt as written, since the practice needs its exact wording:

`Review sessions/S15-pca/activity.ipynb as a reviewer who has not seen it before, against that session's notes.md and the course conventions in CLAUDE.md. Do not edit any file. List numbered findings, each naming the cell: (1) anything that would stop Restart and Run All from finishing; (2) any number written in a markdown cell that does not match the output it describes; (3) any figure without units on both axes or without the sentence after it; (4) any scientific choice in a later cell that is not a Parameters constant. For the judgment cell, say only whether parts (a) and (b) each cite a number that appears in the output, and do not suggest wording.`

Expect, after one to two minutes, a numbered list of findings, and for the judgment cell only whether each part cites a number in the output. A finding names a cell by the id stored in the file, such as `s15-13`, which VS Code does not show: find the cell by the words the finding quotes. The review costs five to ten questions' worth of your usage limit. Read each finding against the cell it names before you act on it: a reviewer that has not run your notebook can misread an output, and a finding you reject with a reason is as good an entry as one you fix. Add a markdown cell under this one (hover just below this cell and click **+ Markdown**) holding these lines, and fill them after the review:

```
**Prompt record**

- **My prompt:**
- **The finding I checked, and what the cell shows:**
- **What I did, fixed or rejected, and why:**
```

Your prompt record is the expected source for part (b) of the judgment cell, and the execution point needs it. If the review asks to edit a file or to run a cell, decline: the prompt says not to.

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): as the archaeologist's analyst, which source did your artefact's obsidian come from, CI, CII or neither? Cite your autoscaled T² and Q ratios to each source's limits, your centered call, whether 1 or 3 components would move the autoscaled call, and say what a wrong call would do to the reading of the site's supply: a source credited with a link it did not have, or a link the data carry left unpublished. Part (b): one judgment entry, a finding from the review that you fixed, rejected or verified, and the reason from your notebook. Your prompt record from section 3 is the natural source; the reason has to cite a value from your output, not a general rule. This session's AI practice, named in the notes, is *a fresh-context review*, and an entry that came from using it is the expected one, though any entry grounded in your data counts.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib
import scipy
import sklearn

print(f"Python       {sys.version.split()[0]}")
print(f"NumPy        {np.__version__}")
print(f"pandas       {pd.__version__}")
print(f"SciPy        {scipy.__version__}")
print(f"Matplotlib   {matplotlib.__version__}")
print(f"scikit-learn {sklearn.__version__}")